## Hadoop: Writing and Reading Data with HDFS

This notebook demonstrates how to interact with HDFS (Hadoop Distributed File System) using Python and `hdfs dfs` commands via `docker exec`. We'll cover the fundamental operations of writing and reading data in HDFS.


#### 1. Connect to HDFS via the proxy (HttpFS)

Creates a client pointing to the proxy (HttpFS, port 14000), which acts as a proxy
and resolves DataNode hostnames internally — no subprocess needed for any operation.


In [ ]:
from hdfs import InsecureClient

client = InsecureClient('http://localhost:14000', user='root')
print('Connected to proxy!')

#### 2. Explore HDFS

Lists files and folders in the HDFS root. This operation uses the NameNode directly (no redirect), so it works fine.

In [ ]:
files = client.list('/')
print(f'Files in HDFS root: {files}')

#### 3. Download sample data

Downloads the Titanic dataset directly from GitHub to the `temp/` directory.

In [ ]:
import urllib.request
import os

os.makedirs('../temp', exist_ok=True)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
urllib.request.urlretrieve(url, '../temp/sample_data.csv')
print('File downloaded: ../temp/sample_data.csv')

#### 4. Upload to HDFS via the proxy

Using the proxy (HttpFS), uploads go through a proxy that resolves DataNode hostnames
internally. The `hdfs` library communicates with HDFS as if it were a single endpoint.


In [ ]:
# Upload via proxy (handles DataNode routing internally)
client.upload(
    hdfs_path='/user/root/data.csv',
    local_path='../temp/sample_data.csv',
    overwrite=True,
    permission=775,
)
print('File uploaded successfully! ➔ /user/root/data.csv')

# Verify the file is there
files = client.list('/user/root/')
print(f'Files: {files}')

#### 5. Read data from HDFS

Read the uploaded file directly into a Pandas DataFrame via the proxy.


In [ ]:
import pandas as pd
import io

# Read the file from HDFS via the proxy and load into Pandas
with client.read('/user/root/data.csv') as reader:
    df = pd.read_csv(io.StringIO(reader.read().decode('utf-8')))
df

#### 6. Cleanup (optional)

Removes the locally downloaded CSV file.

In [ ]:
import os
os.remove('../temp/sample_data.csv')
print('Local file removed.')